# Solar EoL Transportation Cost Estimator

#### Install necessary packages

In [1]:
!pip install beautifulsoup4 pandas folium requests ipywidgets geopy

#### Load packages

In [2]:
import pandas as pd
from bs4 import BeautifulSoup
import re
import math
import ipywidgets as widgets
from IPython.display import display
import requests
import folium

#### Extract position of all solar panel power stations

In [3]:
with open('power_stations.txt', 'r', encoding='utf-8') as file:
    soup = BeautifulSoup(file.read(), 'html.parser')

# Find the table containing the data
table = soup.find('table', class_='power-stations-data-table')
data = []

# Loop through every row in the table body
for row in table.find('tbody').find_all('tr'):
    cols = row.find_all('td')
    
    # Ensure the row has the correct number of columns
    if len(cols) >= 8:
        name = cols[0].text.strip()
        size_raw = cols[3].text.strip()
        
        # Extract numbers and decimal points to handle formats like "1,500", "1500 kW", etc.
        try:
            clean_size = re.sub(r'[^\d.]', '', size_raw)
            size = float(clean_size) if clean_size else 0.0
        except ValueError:
            size = 0.0
        
        # Extract the raw link exactly as it is written in your HTML
        link_tag = cols[7].find('a')
        link = link_tag['href'] if link_tag and 'href' in link_tag.attrs else None
        
        lat, lon = None, None
        
        # Use Regex to extract the coordinates directly from the link string
        if link:
            coords_match = re.search(r'@([-]?\d+\.\d+),([-]?\d+\.\d+)', link)
            
            if coords_match:
                lat = float(coords_match.group(1))
                lon = float(coords_match.group(2))
        
        # Append the extracted info to our list
        data.append({
            'Name': name,
            'Size (kW)': size,
            'Latitude': lat,
            'Longitude': lon
        })

# Convert to a Pandas DataFrame
list_ps = pd.DataFrame(data)

# 1 MW = 1000 kW. You can adjust this to 5000 if your supervisor prefers a 5MW threshold.
UTILITY_THRESHOLD_KW = 1000.0 

# Keep only power stations where the size is strictly greater than or equal to the threshold
list_ps = list_ps[list_ps['Size (kW)'] >= UTILITY_THRESHOLD_KW]

# Reset the index so it looks clean after dropping the commercial rows
list_ps = list_ps.reset_index(drop=True)
list_ps

,Name,Size (kW),Latitude,Longitude
0,Abra Mine Camp,7418.8,-24.633050,118.590998
1,Adelaide Resource Recovery Dry Creek,1300.7,-34.827200,138.553300
2,ADSCS GERALDTON,1560.0,-28.689200,114.843200
3,AFS Cadell Balranald,4826.5,-34.771998,143.500577
4,Agnew Solar Project,3962.7,-27.995337,120.511189
...,...,...,...,...
524,Yarranlea Solar Farm,133769.0,-27.716000,151.564100
525,Yatpool Solar Farm,106497.4,-34.392800,142.174200
526,Yongala Solar Farm,2661.1,-33.025879,138.744596
527,Yulara Solar Plant,1825.3,-25.232203,130.991139


#### List of Recycling Sites

In [4]:
list_rs = pd.read_csv('recycling_sites.csv')
list_rs

,Name,Address,Latitude,Longitude
0,Lotus Recycling,"Unit 1/164-170 Barry Rd, Campbellfield VIC 3061",-37.666821,144.961087
1,ElecSome Kilmany,"14A Velore Rd, Kilmany VIC 3851",-38.102452,146.922293
2,ElecSome Keysborough,"67 Naxos Way, Keysborough VIC 3173",-38.022727,145.184209
3,"Sircel E-waste Recycling Facility, Villawood NSW","82 Marple Ave, Villawood NSW 2163",-33.884851,150.981646
4,"Sircel E-waste Recycling Facility, Parkes NSW","55A Brolgan Rd, Parkes NSW 2870",-33.137845,148.155682
5,PanelCycle,"19-23 Fariola St, Silverwater NSW 2128",-33.830921,151.050608
6,SolaCycle,"2/109 Victoria Rd, Drummoyne NSW 2047",-33.846417,151.157124
7,A1 Metal Recycle,"38 Carrington Rd, Guildford NSW 2161",-33.852583,150.976268
8,SOLAREC SA,"221 Hanson Rd, Athol Park SA 5012",-34.838808,138.541794


#### List of Suppliers/Buyers

In [5]:
list_ss = pd.read_excel('suppliers.xlsx')
list_ss

,Name,Location,Material,Latitude,Longitude
0,Pan Pacific Recycling,"10-12 Magnesium Drive, Crestmead, QLD 4132, Au...",Aluminium,-26.215925,152.363336
1,Yennora Copper Recycling,"31 The Promenade, Yennora NSW 2161",Aluminium,-33.864135,150.978121
2,Highett Metal,"283–295 Boundary Rd, Mordialloc VIC 3195",Aluminium,-37.989055,145.106946
3,United Metal Recycling,"16 Clements Ave, Bundoora VIC 3083",Aluminium,-37.702622,145.073205
4,Super Metal Recycling NSW Pty Ltd,"9 Dunheved Cct, St Marys NSW 2760",Aluminium,-33.532936,150.701044
5,Super Metal Recycling,"345 Frankston–Dandenong Rd, Dandenong South VI...",Aluminium,-37.857918,145.148068
6,All Metals Scrap,"24 Manton Rd, Oakleigh South VIC 3167",Aluminium,-37.914473,145.110406
7,Pan Pacific Recycling,"10-12 Magnesium Drive, Crestmead, QLD 4132, Au...",Copper,-26.215925,152.363336
8,Yennora Copper Recycling,"31 The Promenade, Yennora NSW 2161",Copper,-33.864135,150.978121
9,Safari Copper Recycling,"1/108 Newton Rd, Wetherill Park NSW 2164",Copper,-33.843511,150.892562


#### Add more Power Stations, Recycling Sites, or Suppliers

Run the following cell if you want to add more power stations, recycling sites, and/or suppliers.

In [6]:
# Ask user what they want to add
print("What do you want to add?")
print("1. Power station")
print("2. Recycling site")
print("3. Supplier")
print("4. I don't want to add anything")

choice = input("Enter your choice (1, 2, 3, or 4): ")

# 1. Power station
if choice == '1':
    name = input("Enter Power Station name: ")
    size = float(input("Enter size (kW): "))
    lat = float(input("Enter Latitude: "))
    lon = float(input("Enter Longitude: "))
    
    # Add to list_ps
    new_data = pd.DataFrame([{"Name": name, "Size (kW)": size, "Latitude": lat, "Longitude": lon}])
    list_ps = pd.concat([list_ps, new_data], ignore_index=True)
    print(f"\nSuccessfully added Power Station: {name}")

# 2. Recycling site
elif choice == '2':
    name = input("Enter Recycling Site name: ")
    address = input("Enter Address: ")
    lat = float(input("Enter Latitude: "))
    lon = float(input("Enter Longitude: "))
    
    # Add to list_rs
    new_data = pd.DataFrame([{"Name": name, "Address": address, "Latitude": lat, "Longitude": lon}])
    list_rs = pd.concat([list_rs, new_data], ignore_index=True)
    print(f"\nSuccessfully added Recycling Site: {name}")

# 3. Supplier
elif choice == '3':
    name = input("Enter Supplier name: ")
    address = input("Enter Address: ")
    
    # Simulating a dropdown box with a numbered menu
    print("\nSelect the Material they supply:")
    print("1. Aluminium")
    print("2. Copper")
    print("3. Glass")
    print("4. Plastic/EVA")
    print("5. Silicon")
    print("6. Silver")
    
    # Dictionary to map numbers to the actual material strings
    material_options = {
        "1": "Aluminium", 
        "2": "Copper", 
        "3": "Glass", 
        "4": "Plastic/EVA", 
        "5": "Silicon", 
        "6": "Silver"
    }
    
    mat_choice = input("Enter the number of the material (1-6): ")
    
    # Get the material based on choice, default to Unknown if they type something wrong
    material = material_options.get(mat_choice, "Unknown")
    
    lon = float(input("Enter Longitude: "))
    lat = float(input("Enter Latitude: "))
    
    # Add to list_ss
    new_data = pd.DataFrame([{"Name": name, "Address": address, "Material": material, "Longitude": lon, "Latitude": lat}])
    list_ss = pd.concat([list_ss, new_data], ignore_index=True)
    print(f"\nSuccessfully added Supplier: {name} (Material: {material})")

# 4. Skip
elif choice == '4':
    print("\nNo new data added.")
    
else:
    print("Invalid choice. Please run the code again and enter 1, 2, or 3.")

What do you want to add?
1. Power station
2. Recycling site
3. Supplier
4. I don't want to add anything


Enter your choice (1, 2, 3, or 4):  4



No new data added.


## User Input for Calculation

### Choose a Power Station

In [7]:
# Clean the data
list_ps_clean = list_ps.dropna(subset=['Name', 'Latitude', 'Longitude']).drop_duplicates(subset=['Name'])

# Create a dictionary to quickly map Name -> {Latitude, Longitude}
places_dict = list_ps_clean.set_index('Name')[['Latitude', 'Longitude']].to_dict('index')

# Extract all names as a list of strings for the dropdown
all_places = [str(name) for name in places_dict.keys()]

# Create variables to store the coordinates
ps_lat = None
ps_lon = None

# Create the Searchable Dropdown
place_selector = widgets.Combobox(
    placeholder='Start typing a place name...',
    options=all_places,
    description='Location:',
    ensure_option=True,
    disabled=False,
    layout=widgets.Layout(width='400px')
)

# Create an output area for visual feedback
output = widgets.Output()

# Define the function that runs when a user selects a place
def on_place_change(change):
    global ps_lat, ps_lon 
    output.clear_output()
    selected_ps = change['new']
    
    # Check if the selection is valid
    if selected_ps in places_dict:
        # Extract and save coordinates
        ps_lat = places_dict[selected_ps]['Latitude']
        ps_lon = places_dict[selected_ps]['Longitude']
        
        # Display the result to the user
        with output:
            print(f"Power Station Selected: {selected_ps}")
            print(f"Latitude:  {ps_lat}")
            print(f"Longitude: {ps_lon}")

# Attach the function to the combobox
place_selector.observe(on_place_change, names='value')

# Display the interactive widget
display(place_selector, output)

Combobox(value='', description='Location:', ensure_option=True, layout=Layout(width='400px'), options=('Abra M…

Output()

### How many kW/kg/number of panels user want to recycle?

In [8]:
# Initialise variables globally so other cells can access them
total_kw_min = None
total_kw_max = None
total_panels_min = None
total_panels_max = None
total_kg_min = None
total_kg_max = None
choice = None
value_min = None
value_max = None
kw_per_panel = None
kg_per_panel = None

def convert_solar_data():
    # Tell the function to use the global variables instead of creating local ones
    global choice, value_min, value_max, total_kw_min, total_kw_max
    global total_panels_min, total_panels_max, total_kg_min, total_kg_max
    global kw_per_panel, kg_per_panel
    
    print("--- Solar Module Assumption Setup ---")
    try:
        kw_input = input("Enter the capacity per panel in kW (e.g., 0.300): ")
        kw_per_panel = float(kw_input)
        
        kg_input = input("Enter the weight per panel in kg (e.g., 18.0): ")
        kg_per_panel = float(kg_input)
    except ValueError:
        print("Invalid input. Please enter numerical values for the assumptions.")
        return

    print("\n--- Solar Module Unit Converter ---")
    print("Select the unit to input your data:")
    print("1: Kilowatts (kW)")
    print("2: Kilograms (kg)")
    print("3: Number of Panels")
    
    choice = input("Enter 1, 2, or 3: ")
    
    if choice not in ['1', '2', '3']:
        print("Invalid selection. Please restart.")
        return

    try:
        value_min = float(input("Enter the MINIMUM value of your range: "))
        value_max = float(input("Enter the MAXIMUM value of your range: "))
        
        # Quick safety check in case the user inputs the larger number first
        if value_min > value_max:
            value_min, value_max = value_max, value_min
            
    except ValueError:
        print("Invalid number. Please enter numerical values.")
        return
    
    # Conversion Logic using the user's custom assumptions
    if choice == '1': # User inputted kW
        total_kw_min, total_kw_max = value_min, value_max
        
        total_panels_min = total_kw_min / kw_per_panel
        total_panels_max = total_kw_max / kw_per_panel
        
        total_kg_min = total_panels_min * kg_per_panel
        total_kg_max = total_panels_max * kg_per_panel
        
    elif choice == '2': # User inputted kg
        total_kg_min, total_kg_max = value_min, value_max
        
        total_panels_min = total_kg_min / kg_per_panel
        total_panels_max = total_kg_max / kg_per_panel
        
        total_kw_min = total_panels_min * kw_per_panel
        total_kw_max = total_panels_max * kw_per_panel
        
    elif choice == '3': # User inputted Number of Panels
        total_panels_min, total_panels_max = value_min, value_max
        
        total_kw_min = total_panels_min * kw_per_panel
        total_kw_max = total_panels_max * kw_per_panel
        
        total_kg_min = total_panels_min * kg_per_panel
        total_kg_max = total_panels_max * kg_per_panel

    # Output Results
    print("\n--- Converted Range Values ---")
    print(f"Capacity:      {total_kw_min:,.2f} - {total_kw_max:,.2f} kW")
    print(f"Total Mass:    {total_kg_min:,.2f} - {total_kg_max:,.2f} kg")
    print(f"Total Panels:  {total_panels_min:,.0f} - {total_panels_max:,.0f} panels")

if __name__ == "__main__":
    convert_solar_data()

--- Solar Module Assumption Setup ---


Enter the capacity per panel in kW (e.g., 0.300):  0.3
Enter the weight per panel in kg (e.g., 18.0):  18



--- Solar Module Unit Converter ---
Select the unit to input your data:
1: Kilowatts (kW)
2: Kilograms (kg)
3: Number of Panels


Enter 1, 2, or 3:  1
Enter the MINIMUM value of your range:  1000
Enter the MAXIMUM value of your range:  2000



--- Converted Range Values ---
Capacity:      1,000.00 - 2,000.00 kW
Total Mass:    60,000.00 - 120,000.00 kg
Total Panels:  3,333 - 6,667 panels


### User want to add Recycling Site?

In [9]:
print("--- Optional Custom Recycling Site ---")
print("If you want to estimate the transportation cost from a specific recycling site, enter the details below.")
print("If not, just press Enter for all prompts to automatically find the closest site.\n")

custom_rs_name = input("Enter Recycling Site Name (or press Enter to skip): ").strip()
custom_rs_address = input("Enter Recycling Site Address (or press Enter to skip): ").strip()
custom_rs_lat_input = input("Enter Latitude (e.g., -33.8688) (or press Enter to skip): ").strip()
custom_rs_lon_input = input("Enter Longitude (e.g., 151.2093) (or press Enter to skip): ").strip()

# Validate inputs and set a flag for the next cell
use_custom_rs = False

if custom_rs_name and custom_rs_lat_input and custom_rs_lon_input:
    try:
        custom_rs_lat = float(custom_rs_lat_input)
        custom_rs_lon = float(custom_rs_lon_input)
        use_custom_rs = True
        print(f"\nCustom site '{custom_rs_name}' registered successfully.")
    except ValueError:
        print("\nInvalid coordinates entered. Falling back to automatic closest site detection.")
else:
    print("\nNo custom site provided. The calculator will use the closest recycling site.")

--- Optional Custom Recycling Site ---
If you want to estimate the transportation cost from a specific recycling site, enter the details below.
If not, just press Enter for all prompts to automatically find the closest site.



Enter Recycling Site Name (or press Enter to skip):  My r
Enter Recycling Site Address (or press Enter to skip):  123 rs
Enter Latitude (e.g., -33.8688) (or press Enter to skip):  -33.8688
Enter Longitude (e.g., 151.2093) (or press Enter to skip):  151.2093



Custom site 'My r' registered successfully.


## Conventional Recycling

### Number of Truck Needed

In [10]:
# Truck specifications
truck_volume_limit = 28 # cubic meters
truck_weight_limit = 7000 # kg
truck_w, truck_l, truck_h = 2.2, 6.2, 2.4 # meters

# Module specifications
module_capacity_w = 300 # Watts
module_w, module_l, module_h = 1.0, 1.7, 0.035 # meters
module_weight = (module_capacity_w / 1000) * 60 # 18.0 kg
module_volume = module_w * module_l * module_h # 0.0595 cubic meters

print("Capacity of the truck:")
# Number of panels limit by volume
panels_by_volume = int(truck_volume_limit // module_volume)
print(f"Limit by pure volume: {panels_by_volume} panels")

# Number of panels limit by weight
panels_by_weight = int(truck_weight_limit // module_weight)
print(f"Limit by pure weight: {panels_by_weight} panels")

# Number of panels limit by physical space
stacks_wide = int(truck_w // module_w)
stacks_long = int(truck_l // module_l)
panels_high = int(truck_h // module_h)

panels_by_space = stacks_wide * stacks_long * panels_high
print(f"Limit by physical space: {panels_by_space} panels")

# Store the limits in a dictionary to link the names to the calculated values
limits = {
    "pure volume": panels_by_volume,
    "pure weight": panels_by_weight,
    "physical space": panels_by_space
}

limiting_factor = min(limits, key=limits.get)
max_panels = limits[limiting_factor]
print(f"\nConclusion: \nThe truck is limited by {limiting_factor}, it can carry a maximum of {max_panels} panels.\n")

# Determine the unit string based on the user's choice from the previous cell
if choice == '1':
    unit = "kW"
elif choice == '2':
    unit = "kg"
elif choice == '3':
    unit = "panels"
else:
    unit = "units" # Fallback just in case

# Calculate the number of trucks needed for both ends of the range (rounding up)
trucks_needed_min = math.ceil(total_panels_min / max_panels)
trucks_needed_max = math.ceil(total_panels_max / max_panels)

# Print the final output as a range
print(f"If the user wants to recycle {value_min:g} - {value_max:g} {unit} of solar EoL,")
print(f"they will need {trucks_needed_min} - {trucks_needed_max} trucks.")

Capacity of the truck:
Limit by pure volume: 470 panels
Limit by pure weight: 388 panels
Limit by physical space: 408 panels

Conclusion: 
The truck is limited by pure weight, it can carry a maximum of 388 panels.

If the user wants to recycle 1000 - 2000 kW of solar EoL,
they will need 9 - 18 trucks.


### Estimate Driving Distance and Optimal Route to Nearest Recycling Site

In [11]:
# Define the function to get driving distance and route geometry from OSRM
def get_osrm_driving_data(lat1, lon1, lat2, lon2):
    # OSRM API expects coordinates in longitude,latitude order
    url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=full&geometries=geojson"
    
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if data['code'] == 'Ok':
            # Extract distance (OSRM returns meters, we convert to km)
            distance_km = data['routes'][0]['distance'] / 1000
            # Extract the shape of the route for mapping
            route_geometry = data['routes'][0]['geometry'] 
            return distance_km, route_geometry
            
    # If the API fails or no route is found, return infinity
    return float('inf'), None

# Initialise tracking variables
target_site = None
driving_distance = float('inf')
best_route_geometry = None

if use_custom_rs:
    # ---------------------------------------------------------
    # CASE 1: User inputted a specific recycling site
    # ---------------------------------------------------------
    print(f"Calculating route to custom site: {custom_rs_name}...")
    dist_km, geometry = get_osrm_driving_data(ps_lat, ps_lon, custom_rs_lat, custom_rs_lon)
    
    driving_distance = dist_km
    best_route_geometry = geometry
    
    # Create a dictionary mimicking a dataframe row for seamless mapping below
    target_site = {
        'Name': custom_rs_name,
        'Address': custom_rs_address if custom_rs_address else "Custom Input Address",
        'Latitude': custom_rs_lat,
        'Longitude': custom_rs_lon
    }

else:
    # ---------------------------------------------------------
    # CASE 2: Find the closest site from the dataframe
    # ---------------------------------------------------------
    print("Finding the closest recycling site from the database...")
    for index, site in list_rs.iterrows():
        site_lat = site['Latitude']
        site_lon = site['Longitude']
        
        # Calculate driving distance
        dist_km, geometry = get_osrm_driving_data(ps_lat, ps_lon, site_lat, site_lon)
        
        # Update if this driving route is shorter than our current best
        if dist_km < driving_distance:
            driving_distance = dist_km
            target_site = site
            best_route_geometry = geometry

# Ensure a valid route was found before attempting to map
if target_site and driving_distance != float('inf'):
    rs_lat = target_site['Latitude']
    rs_lon = target_site['Longitude']

    # Output the results
    print("-" * 40)
    print(f"Destination Hub: {target_site['Name']}")
    print(f"Address: {target_site['Address']}")
    print(f"Driving Distance: {driving_distance:.2f} km")
    print("-" * 40)

    # Create the Map
    mid_lat = (ps_lat + rs_lat) / 2
    mid_lon = (ps_lon + rs_lon) / 2
    route_map = folium.Map(location=[mid_lat, mid_lon], zoom_start=10)

    # Add Power Station Marker
    folium.Marker(
        location=[ps_lat, ps_lon],
        popup="Selected Power Station",
        icon=folium.Icon(color="blue", icon="bolt", prefix='fa')
    ).add_to(route_map)

    # Add Recycling Hub Marker
    folium.Marker(
        location=[rs_lat, rs_lon],
        popup=f"Recycling Hub: {target_site['Name']}",
        icon=folium.Icon(color="green", icon="recycle", prefix='fa')
    ).add_to(route_map)

    # Draw the actual road route using the GeoJSON from OSRM
    if best_route_geometry:
        folium.GeoJson(
            best_route_geometry,
            name="Driving Route",
            style_function=lambda feature: {
                'color': '#FF0000',
                'weight': 4,
                'opacity': 0.8
            },
            tooltip=f"Driving distance: {driving_distance:.2f} km"
        ).add_to(route_map)

    # Display the map in Jupyter Notebook
    display(route_map)
else:
    print("Error: Could not calculate a valid driving route.")

Calculating route to custom site: My r...
----------------------------------------
Destination Hub: My r
Address: 123 rs
Driving Distance: 4277.50 km
----------------------------------------


### Estimate Distance and Route to CLosest Suppliers for 6 Materials

In [16]:
# Initialise a dictionary to store the best supplier and route for each material
best_suppliers = {}

# Get the unique list of materials from the dataset (assuming there are 6)
materials = list_ss['Material'].unique()

# Define some distinct colors for the 6 different routes on the map
route_colors = ['#FF0000', '#0000FF', '#008000', '#800080', '#FFA500', '#00FFFF']

print("Finding the closest suppliers for each material...\n")

# Loop through each unique material
for i, material in enumerate(materials):
    # Filter the DataFrame for suppliers of this specific material
    material_suppliers = list_ss[list_ss['Material'] == material]
    
    min_dist = float('inf')
    best_supp = None
    best_geom = None
    
    # Loop through suppliers of this material to find the closest one
    for index, supp in material_suppliers.iterrows():
        supp_lat = supp['Latitude']
        supp_lon = supp['Longitude']
        
        # Call the existing OSRM function (using rs_lat and rs_lon from previous cell)
        dist, geom = get_osrm_driving_data(rs_lat, rs_lon, supp_lat, supp_lon)
        
        # Update if it's the closest one found so far for this material
        if dist < min_dist:
            min_dist = dist
            best_supp = supp
            best_geom = geom
            
    # Save the best result for this material
    if best_supp is not None:
        best_suppliers[material] = {
            'supplier': best_supp,
            'distance': min_dist,
            'geometry': best_geom,
            'color': route_colors[i % len(route_colors)]
        }

# Output the Results
print("CLOSEST SUPPLIERS IDENTIFIED:\n" + "-"*40)
for mat, data in best_suppliers.items():
    supp = data['supplier']
    print(f"Material: {mat}")
    print(f"Company: {supp['Name']}")
    print(f"Address: {supp['Location']}")
    print(f"Driving Distance: {data['distance']:.2f} km\n")

# Create the Map
# Center the map on the Recycling Site
suppliers_map = folium.Map(location=[rs_lat, rs_lon], zoom_start=9)

# Add Recycling Site Marker (Origin)
folium.Marker(
    location=[rs_lat, rs_lon],
    popup="Recycling Site (Origin)",
    icon=folium.Icon(color="green", icon="recycle", prefix='fa')
).add_to(suppliers_map)

# Loop through the best suppliers to add markers and routes to the map
for mat, data in best_suppliers.items():
    supp = data['supplier']
    geom = data['geometry']
    color = data['color']
    
    # Add Supplier Marker
    folium.Marker(
        location=[supp['Latitude'], supp['Longitude']],
        popup=f"<b>{supp['Name']}</b><br>Material: {mat}",
        icon=folium.Icon(color="orange", icon="industry", prefix='fa')
    ).add_to(suppliers_map)
    
    # Add Driving Route
    if geom:
        folium.GeoJson(
            geom,
            name=f"{mat} Route",
            # We use color=color in the lambda to ensure Python binds the current color in the loop correctly
            style_function=lambda feature, c=color: {
                'color': c,
                'weight': 4,
                'opacity': 0.8
            },
            tooltip=f"{mat} Route: {data['distance']:.2f} km"
        ).add_to(suppliers_map)

# Add layer control to toggle routes on/off
folium.LayerControl().add_to(suppliers_map)

# Display the map
suppliers_map

Finding the closest suppliers for each material...

CLOSEST SUPPLIERS IDENTIFIED:
----------------------------------------
Material: Aluminium
Company: Super Metal Recycling NSW Pty Ltd
Address: 9 Dunheved Cct, St Marys NSW 2760
Driving Distance: 293.96 km

Material: Copper
Company: Safari Copper Recycling
Address: 1/108 Newton Rd, Wetherill Park NSW 2164
Driving Distance: 335.58 km

Material: Glass
Company: Sydney Glass Pty Ltd
Address: 30 Short St, Bankstown NSW 2211
Driving Distance: 361.70 km

Material: Plastic / EVA
Company: Polyflor Australia NSW
Address: 1/2 Southridge St, Eastern Creek NSW 2766
Driving Distance: 322.54 km

Material: Silicon
Company: Soudal Australia
Address: 75 Owen St, Glendenning NSW 2761
Driving Distance: 320.27 km

Material: Silver
Company: Ore Metal
Address: 115 Booth St, Annandale NSW 2038
Driving Distance: 358.81 km



### Estimate Transportation Cost

In [17]:
# Full Route: Recycling Site -> Power Station -> Recycling Site -> Suppliers -> Recycling Site
# Our assumptions
diesel_price_per_litre = 2.25
fuel_efficiency_L_per_100km = 17
fuel_efficiency_L_per_km = fuel_efficiency_L_per_100km / 100

# Path 1: Recycling Site -> Power Station -> Recycling Site
# Fuel Cost
round_trip_distance_km = min_driving_distance * 2
fuel_used_per_truck = round_trip_distance_km * fuel_efficiency_L_per_km
cost_per_truck = fuel_used_per_truck * diesel_price_per_litre

total_fuel_cost_1_min = cost_per_truck * trucks_needed_min
total_fuel_cost_1_max = cost_per_truck * trucks_needed_max

# Labour Cost
driver_per_truck = 1
helper_per_truck = 2
driver_hourly_rate = 42
helper_hourly_rate = 45    
average_speed_kmh = 45     # Assumed constant speed in km/hr
onsite_work_hours = 4      # Time spent loading/unloading

driving_time_hours = round_trip_distance_km / average_speed_kmh
combined_hourly_rate_path1 = (driver_per_truck * driver_hourly_rate) + (helper_per_truck * helper_hourly_rate)
hours_per_truck_path1 = driving_time_hours + onsite_work_hours

total_labour_cost_1_min = combined_hourly_rate_path1 * hours_per_truck_path1 * trucks_needed_min
total_labour_cost_1_max = combined_hourly_rate_path1 * hours_per_truck_path1 * trucks_needed_max

# Total Cost Path 1
total_cost_1_min = total_fuel_cost_1_min + total_labour_cost_1_min
total_cost_1_max = total_fuel_cost_1_max + total_labour_cost_1_max

print("PATH 1: Recycling Site -> Power Station -> Recycling Site")
print("-" * 55)
print(f"Round-trip distance per truck: {round_trip_distance_km:.2f} km")
print(f"TOTAL FLEET FUEL COST: ${total_fuel_cost_1_min:,.2f} - ${total_fuel_cost_1_max:,.2f} AUD")
print(f"TOTAL FLEET LABOUR COST: ${total_labour_cost_1_min:,.2f} - ${total_labour_cost_1_max:,.2f} AUD")
print(f"TOTAL PATH 1 COST: ${total_cost_1_min:,.2f} - ${total_cost_1_max:,.2f} AUD\n")


# Path 2: Recycling Site -> Suppliers -> Recycling Site
# Material composition assumptions
material_composition = {
    'Glass': 0.72,
    'Aluminium': 0.15,
    'Plastic / EVA': 0.07,
    'Silicon': 0.025,
    'Copper': 0.008,
    'Silver': 0.0003
}

total_fuel_cost_2_min = 0
total_fuel_cost_2_max = 0
total_labour_cost_2_min = 0
total_labour_cost_2_max = 0

print("PATH 2: Recycling Site -> Suppliers -> Recycling Site")
print("-" * 55)

# Loop through the dictionary created in your Folium map code
for mat, data in best_suppliers.items():
    if mat not in material_composition:
        print(f"Warning: {mat} not in composition dictionary. Skipping.")
        continue
        
    # Calculate weight ranges
    mat_kg_min = total_kg_min * material_composition[mat]
    mat_kg_max = total_kg_max * material_composition[mat]
    
    # Calculate truck ranges
    trucks_for_mat_min = math.ceil(mat_kg_min / 7000)  # 7000 kg capacity per truck
    trucks_for_mat_max = math.ceil(mat_kg_max / 7000)
    
    if trucks_for_mat_max == 0:
        continue # Skip if the maximum weight still yields 0 trucks
        
    # Get route distance (from OSRM output stored in best_suppliers)
    mat_one_way_dist = data['distance']
    mat_round_trip_dist = mat_one_way_dist * 2
    
    # Fuel Cost for this material
    mat_fuel_used = mat_round_trip_dist * fuel_efficiency_L_per_km
    mat_fuel_cost_min = mat_fuel_used * diesel_price_per_litre * trucks_for_mat_min
    mat_fuel_cost_max = mat_fuel_used * diesel_price_per_litre * trucks_for_mat_max
    
    total_fuel_cost_2_min += mat_fuel_cost_min
    total_fuel_cost_2_max += mat_fuel_cost_max
    
    # Labour Cost for this material
    mat_driving_hours = mat_round_trip_dist / average_speed_kmh
    mat_onsite_hours = 2 # 2 hours loading/unloading
    
    # Cost = combined hourly rate * total hours per truck * number of trucks
    combined_hourly_rate_path2 = (driver_per_truck * driver_hourly_rate) + (helper_per_truck * helper_hourly_rate)
    hours_per_truck_path2 = mat_driving_hours + mat_onsite_hours
    
    mat_labour_cost_min = combined_hourly_rate_path2 * hours_per_truck_path2 * trucks_for_mat_min
    mat_labour_cost_max = combined_hourly_rate_path2 * hours_per_truck_path2 * trucks_for_mat_max
    
    total_labour_cost_2_min += mat_labour_cost_min
    total_labour_cost_2_max += mat_labour_cost_max
    
    print(f"Material: {mat} ({mat_kg_min:,.2f} - {mat_kg_max:,.2f} kg)")
    print(f"  Trucks required: {trucks_for_mat_min} - {trucks_for_mat_max}")
    print(f"  Round-trip distance: {mat_round_trip_dist:.2f} km")
    print(f"  Fuel: ${mat_fuel_cost_min:,.2f} - ${mat_fuel_cost_max:,.2f} | Labour: ${mat_labour_cost_min:,.2f} - ${mat_labour_cost_max:,.2f}\n")

total_cost_2_min = total_fuel_cost_2_min + total_labour_cost_2_min
total_cost_2_max = total_fuel_cost_2_max + total_labour_cost_2_max

print(f"TOTAL PATH 2 FUEL COST: ${total_fuel_cost_2_min:,.2f} - ${total_fuel_cost_2_max:,.2f} AUD")
print(f"TOTAL PATH 2 LABOUR COST: ${total_labour_cost_2_min:,.2f} - ${total_labour_cost_2_max:,.2f} AUD")
print(f"TOTAL PATH 2 COST: ${total_cost_2_min:,.2f} - ${total_cost_2_max:,.2f} AUD\n")

# Full Route Transportation Cost
total_project_cost_min = total_cost_1_min + total_cost_2_min
total_project_cost_max = total_cost_1_max + total_cost_2_max

print("=======================================================")
print(f"FULL ROUTE TOTAL COST (PATH 1 + PATH 2): ${total_project_cost_min:,.2f} - ${total_project_cost_max:,.2f} AUD")
print("=======================================================")

PATH 1: Recycling Site -> Power Station -> Recycling Site
-------------------------------------------------------
Round-trip distance per truck: 602.30 km
TOTAL FLEET FUEL COST: $2,073.41 - $2,303.79 AUD
TOTAL FLEET LABOUR COST: $20,652.69 - $22,947.44 AUD
TOTAL PATH 1 COST: $22,726.11 - $25,251.23 AUD

PATH 2: Recycling Site -> Suppliers -> Recycling Site
-------------------------------------------------------
Material: Aluminium (9,000.00 - 9,594.00 kg)
  Trucks required: 2 - 2
  Round-trip distance: 587.92 km
  Fuel: $449.76 - $449.76 | Labour: $3,977.11 - $3,977.11

Material: Copper (480.00 - 511.68 kg)
  Trucks required: 1 - 1
  Round-trip distance: 671.17 km
  Fuel: $256.72 - $256.72 | Labour: $2,232.75 - $2,232.75

Material: Glass (43,200.00 - 46,051.20 kg)
  Trucks required: 7 - 7
  Round-trip distance: 723.39 km
  Fuel: $1,936.89 - $1,936.89 | Labour: $16,701.69 - $16,701.69

Material: Plastic / EVA (4,200.00 - 4,477.20 kg)
  Trucks required: 1 - 1
  Round-trip distance: 645.0

## Mobile Recycling

### Find Closest Suppliers

In [21]:
# Initialise a dictionary to store the best supplier and route for each material
best_suppliers = {}

# Get the unique list of materials from the dataset (assuming there are 6)
materials = list_ss['Material'].unique()

# Define some distinct colors for the 6 different routes on the map
route_colors = ['#FF0000', '#0000FF', '#008000', '#800080', '#FFA500', '#00FFFF']

print("Finding the closest suppliers for each material...\n")

# Loop through each unique material
for i, material in enumerate(materials):
    # Filter the DataFrame for suppliers of this specific material
    material_suppliers = list_ss[list_ss['Material'] == material]
    
    min_dist = float('inf')
    best_supp = None
    best_geom = None
    
    # Loop through suppliers of this material to find the closest one
    for index, supp in material_suppliers.iterrows():
        supp_lat = supp['Latitude']
        supp_lon = supp['Longitude']
        
        # Call the existing OSRM function using the Power Station coordinates (ps_lat and ps_lon)
        dist, geom = get_osrm_driving_data(ps_lat, ps_lon, supp_lat, supp_lon)
        
        # Update if it's the closest one found so far for this material
        if dist < min_dist:
            min_dist = dist
            best_supp = supp
            best_geom = geom
            
    # Save the best result for this material
    if best_supp is not None:
        best_suppliers[material] = {
            'supplier': best_supp,
            'distance': min_dist,
            'geometry': best_geom,
            'color': route_colors[i % len(route_colors)]
        }

# Output the Results
print("CLOSEST SUPPLIERS IDENTIFIED:\n" + "-"*40)
for mat, data in best_suppliers.items():
    supp = data['supplier']
    print(f"Material: {mat}")
    print(f"Company: {supp['Name']}")
    print(f"Address: {supp['Location']}")
    print(f"Driving Distance: {data['distance']:.2f} km\n")

# Create the Map
# Center the map on the Power Station
suppliers_map = folium.Map(location=[ps_lat, ps_lon], zoom_start=9)

# Add Power Station Marker (Origin)
folium.Marker(
    location=[ps_lat, ps_lon],
    popup="Power Station (Origin)",
    icon=folium.Icon(color="red", icon="bolt", prefix='fa')
).add_to(suppliers_map)

# Loop through the best suppliers to add markers and routes to the map
for mat, data in best_suppliers.items():
    supp = data['supplier']
    geom = data['geometry']
    color = data['color']
    
    # Add Supplier Marker
    folium.Marker(
        location=[supp['Latitude'], supp['Longitude']],
        popup=f"<b>{supp['Name']}</b><br>Material: {mat}",
        icon=folium.Icon(color="orange", icon="industry", prefix='fa')
    ).add_to(suppliers_map)
    
    # Add Driving Route
    if geom:
        folium.GeoJson(
            geom,
            name=f"{mat} Route",
            # We use color=color in the lambda to ensure Python binds the current color in the loop correctly
            style_function=lambda feature, c=color: {
                'color': c,
                'weight': 4,
                'opacity': 0.8
            },
            tooltip=f"{mat} Route: {data['distance']:.2f} km"
        ).add_to(suppliers_map)

# Add layer control to toggle routes on/off
folium.LayerControl().add_to(suppliers_map)

# Display the map
suppliers_map

Finding the closest suppliers for each material...

CLOSEST SUPPLIERS IDENTIFIED:
----------------------------------------
Material: Aluminium
Company: United Metal Recycling
Address: 16 Clements Ave, Bundoora VIC 3083
Driving Distance: 433.87 km

Material: Copper
Company: Safari Copper Recycling
Address: 1/108 Newton Rd, Wetherill Park NSW 2164
Driving Distance: 513.79 km

Material: Glass
Company: Alex Fraser
Address: Level 1, 50 Parkwest Drive, Derrimut, VIC 3026
Driving Distance: 443.67 km

Material: Plastic / EVA
Company: Polyflor Australia VIC
Address: 101 Prosperity Way, Dandenong South VIC 3175
Driving Distance: 468.02 km

Material: Silicon
Company: Admil Sealants & Adhesives
Address: 80/84 Peters Ave, Mulgrave VIC 3170
Driving Distance: 464.46 km

Material: Silver
Company: Palloys
Address: 8/10 Meeks Rd, Marrickville NSW 2204
Driving Distance: 528.76 km



### Estimate Labour and Transport Cost of Mobile System (1 Vehicle)

In [23]:
# ==========================================
# ASSUMPTIONS & RATES
# ==========================================
# Mobile System Assumptions
mobile_parking_distance_km = 120
processing_capacity_kg_per_day = 3000
working_hours_per_day = 8

# Electricity Assumptions
power_consumption_kw = 100 # Assumed power draw of the mobile recycling unit
electricity_price_per_kwh = 0.25 # AUD

# Fuel & Labour Assumptions (Consistent with conventional)
diesel_price_per_litre = 2.25
fuel_efficiency_L_per_km = 17 / 100
average_speed_kmh = 45

driver_hourly_rate = 42
helper_hourly_rate = 45
driver_per_truck = 1
helper_per_truck = 2
combined_hourly_rate = (driver_per_truck * driver_hourly_rate) + (helper_per_truck * helper_hourly_rate)

# ==========================================
# PART 1: MOBILE SYSTEM DEPLOYMENT & OPERATION
# ==========================================
print("PART 1: Mobile System Operation (Parking -> Power Station -> Parking)")
print("-" * 70)

# Calculate Fuel for Mobile Unit (1 vehicle round trip)
mobile_round_trip_km = mobile_parking_distance_km * 2
mobile_fuel_cost = mobile_round_trip_km * fuel_efficiency_L_per_km * diesel_price_per_litre

# Calculate Processing Time
days_on_site_min = math.ceil(total_kg_min / processing_capacity_kg_per_day)
days_on_site_max = math.ceil(total_kg_max / processing_capacity_kg_per_day)

operating_hours_min = days_on_site_min * working_hours_per_day
operating_hours_max = days_on_site_max * working_hours_per_day

# Calculate Electricity Cost
electricity_cost_min = operating_hours_min * power_consumption_kw * electricity_price_per_kwh
electricity_cost_max = operating_hours_max * power_consumption_kw * electricity_price_per_kwh

# Calculate Labour Cost (Driving time + Operating time on site)
mobile_driving_hours = mobile_round_trip_km / average_speed_kmh
mobile_labour_cost_min = combined_hourly_rate * (mobile_driving_hours + operating_hours_min)
mobile_labour_cost_max = combined_hourly_rate * (mobile_driving_hours + operating_hours_max)

# Total Phase 1 Cost
total_mobile_op_cost_min = mobile_fuel_cost + electricity_cost_min + mobile_labour_cost_min
total_mobile_op_cost_max = mobile_fuel_cost + electricity_cost_max + mobile_labour_cost_max

print(f"Estimated Days on Site: {days_on_site_min} - {days_on_site_max} days")
print(f"Electricity Cost: ${electricity_cost_min:,.2f} - ${electricity_cost_max:,.2f} AUD")
print(f"Fuel Cost: ${mobile_fuel_cost:,.2f} AUD (Fixed 240km trip)")
print(f"Labour Cost: ${mobile_labour_cost_min:,.2f} - ${mobile_labour_cost_max:,.2f} AUD")
print(f"TOTAL DEPLOYMENT COST: ${total_mobile_op_cost_min:,.2f} - ${total_mobile_op_cost_max:,.2f} AUD\n")


# ==========================================
# PART 2: TRUCKS TO SUPPLIERS
# ==========================================
print("PART 2: Material Transport (Parking -> PS -> Supplier -> PS -> Parking)")
print("-" * 70)

material_composition = {
    'Glass': 0.72, 'Aluminium': 0.15, 'Plastic / EVA': 0.07,
    'Silicon': 0.025, 'Copper': 0.008, 'Silver': 0.0003
}

total_supp_fuel_cost_min = 0
total_supp_fuel_cost_max = 0
total_supp_labour_cost_min = 0
total_supp_labour_cost_max = 0

for mat, data in best_suppliers.items():
    if mat not in material_composition:
        continue
        
    mat_kg_min = total_kg_min * material_composition[mat]
    mat_kg_max = total_kg_max * material_composition[mat]
    
    trucks_for_mat_min = math.ceil(mat_kg_min / 7000)
    trucks_for_mat_max = math.ceil(mat_kg_max / 7000)
    
    if trucks_for_mat_max == 0:
        continue
        
    # Truck distance = Parking to PS (120) + PS to Supp + Supp to PS + PS to Parking (120)
    mat_one_way_dist = data['distance']
    truck_round_trip_dist = (mobile_parking_distance_km * 2) + (mat_one_way_dist * 2)
    
    # Fuel Calculation
    fuel_per_truck = truck_round_trip_dist * fuel_efficiency_L_per_km * diesel_price_per_litre
    total_supp_fuel_cost_min += fuel_per_truck * trucks_for_mat_min
    total_supp_fuel_cost_max += fuel_per_truck * trucks_for_mat_max
    
    # Labour Calculation (Driving + 4 hours total for loading/unloading)
    truck_driving_hours = truck_round_trip_dist / average_speed_kmh
    truck_labour_per_truck = combined_hourly_rate * (truck_driving_hours + 4)
    
    total_supp_labour_cost_min += truck_labour_per_truck * trucks_for_mat_min
    total_supp_labour_cost_max += truck_labour_per_truck * trucks_for_mat_max
    
    print(f"Material: {mat} ({mat_kg_min:,.2f} - {mat_kg_max:,.2f} kg)")
    print(f"  Trucks required: {trucks_for_mat_min} - {trucks_for_mat_max}")
    print(f"  Route per truck: {truck_round_trip_dist:.2f} km")

total_supp_cost_min = total_supp_fuel_cost_min + total_supp_labour_cost_min
total_supp_cost_max = total_supp_fuel_cost_max + total_supp_labour_cost_max

print(f"\nTOTAL TRANSPORT FUEL COST: ${total_supp_fuel_cost_min:,.2f} - ${total_supp_fuel_cost_max:,.2f} AUD")
print(f"TOTAL TRANSPORT LABOUR COST: ${total_supp_labour_cost_min:,.2f} - ${total_supp_labour_cost_max:,.2f} AUD")
print(f"TOTAL TRANSPORT COST: ${total_supp_cost_min:,.2f} - ${total_supp_cost_max:,.2f} AUD\n")

# ==========================================
# TOTAL PROJECT COST
# ==========================================
total_mobile_project_cost_min = total_mobile_op_cost_min + total_supp_cost_min
total_mobile_project_cost_max = total_mobile_op_cost_max + total_supp_cost_max

print("=======================================================")
print(f"FULL MOBILE RECYCLING TOTAL COST: ${total_mobile_project_cost_min:,.2f} - ${total_mobile_project_cost_max:,.2f} AUD")
print("=======================================================")

PART 1: Mobile System Operation (Parking -> Power Station -> Parking)
----------------------------------------------------------------------
Estimated Days on Site: 20 - 22 days
Electricity Cost: $4,000.00 - $4,400.00 AUD
Fuel Cost: $91.80 AUD (Fixed 240km trip)
Labour Cost: $21,824.00 - $23,936.00 AUD
TOTAL DEPLOYMENT COST: $25,915.80 - $28,427.80 AUD

PART 2: Material Transport (Parking -> PS -> Supplier -> PS -> Parking)
----------------------------------------------------------------------
Material: Aluminium (9,000.00 - 9,594.00 kg)
  Trucks required: 2 - 2
  Route per truck: 1107.73 km
Material: Copper (480.00 - 511.68 kg)
  Trucks required: 1 - 1
  Route per truck: 1267.58 km
Material: Glass (43,200.00 - 46,051.20 kg)
  Trucks required: 7 - 7
  Route per truck: 1127.34 km
Material: Plastic / EVA (4,200.00 - 4,477.20 kg)
  Trucks required: 1 - 1
  Route per truck: 1176.04 km
Material: Silicon (1,500.00 - 1,599.00 kg)
  Trucks required: 1 - 1
  Route per truck: 1168.93 km
Material

### Estimate Worth of EoL Waste (in $)

In [20]:
# Based on our assumptions
panel_waste_worth_min = total_kg_min * 0.2 # AUD
panel_waste_worth_max = total_kg_max * 0.2 # AUD

print(f"Estimate total worth of recycled panels: ${panel_waste_worth_min:,.2f} - ${panel_waste_worth_max:,.2f} AUD")

Estimate total worth of recycled panels: $12,000.00 - $12,792.00 AUD
